# PlantSwarm — Colab (single notebook)

**Before you run:** set **Runtime → Change runtime type → GPU** (T4/L4/A100). Models will not load correctly on CPU-only for the VL demo.

**Run cells in order (§1 → §6).** Skipping §1 breaks Hugging Face cache paths; skipping §4 leaves `PLANTSWARM_BACKBONE` undefined for §5.

**Sync:** §1 sets `HF_HOME` / `HUGGINGFACE_HUB_CACHE` / `TRANSFORMERS_CACHE` and `TFDS_DATA_DIR` so every download (TFDS + Hub) lands in one place (Drive if `USE_DRIVE`).

**Optional:** In Colab, open **Secrets** (key icon) and add `HF_TOKEN` if any model is gated.

**Pipeline:** (1) paths & caches → (2) pip once → (3) clone repo → (4) **model IDs + `snapshot_download` + Transformers load** → (5) YAML → (6) scripts. HTTP inference still needs **vLLM** (or compatible API) at `VLLM_BASE_URL` for `run_plantswarm`.


## 1 — Google Drive, results dir & Hugging Face cache

Set `USE_DRIVE = True` to persist **results**, **TFDS**, and **Hugging Face hub cache** on Drive (recommended).


In [ ]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import os
import sys
from pathlib import Path
from datetime import datetime, timezone

os.environ.setdefault("PYTHONIOENCODING", "utf-8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

USE_DRIVE = True  # False → keep caches under /content only

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_BASE = Path("/content/drive/MyDrive/PlantSwarm_runs")
else:
    OUT_BASE = Path("/content/socioswarm_out")

OUT_BASE.mkdir(parents=True, exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
RESULTS_DIR = (OUT_BASE / RUN_ID).resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# TFDS (Plant Village) — same env as local loaders / YAML
TFDS_PARENT = OUT_BASE / "tensorflow_datasets"
TFDS_PARENT.mkdir(parents=True, exist_ok=True)
os.environ["TFDS_DATA_DIR"] = str(TFDS_PARENT)

# Hugging Face — all Hub downloads use this cache (must run before §4)
HF_HOME = OUT_BASE / "huggingface"
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_HOME / "transformers")

# Optional gated models (Colab Secrets: HF_TOKEN)
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("HF_TOKEN: loaded from Colab Secrets")
except Exception:
    pass

print("RESULTS_DIR       =", RESULTS_DIR)
print("TFDS_DATA_DIR     =", os.environ["TFDS_DATA_DIR"])
print("HF_HOME           =", os.environ["HF_HOME"])
print("HUGGINGFACE_HUB_CACHE =", os.environ["HUGGINGFACE_HUB_CACHE"])
print("TRANSFORMERS_CACHE    =", os.environ["TRANSFORMERS_CACHE"])


## 2 — Install **all** Python dependencies (one cell)

Includes torch, transformers, AutoGen, TFDS, etc. **No separate model pip** elsewhere.


In [ ]:
# -*- coding: utf-8 -*-
import subprocess
import sys


def pip_install(args: list[str]) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)


pip_install(["-U", "pip", "setuptools", "wheel"])
pip_install(["huggingface_hub"])
pip_install(["torch", "transformers", "accelerate", "safetensors"])
pip_install(["pyarrow", "pandas", "numpy", "datasets", "python-dotenv"])
pip_install(["Pillow", "scikit-learn", "scipy", "statsmodels"])
pip_install(["tqdm", "pyyaml", "requests"])
pip_install(["autogen-agentchat>=0.4.0", "autogen-ext[openai]>=0.4.0"])
pip_install(["tensorflow-cpu>=2.15", "tensorflow-datasets>=4.9", "importlib_resources>=6.0"])
pip_install(["protobuf>=5.29.3,<6"])
# Optional: pip_install(["vllm>=0.4.0"])  # GPU; for OpenAI-compatible server in-session

print("pip installs finished.")



## 3 — Clone or locate the repository

Set `GIT_REPO` to clone; otherwise use `/content/socioswarm_repo` after upload.


In [ ]:
# -*- coding: utf-8 -*-
import os
import subprocess
import sys
from pathlib import Path

GIT_REPO = "https://github.com/tirtho149/PlantSwarm.git"  # set "" if you upload the repo manually
BRANCH = "main"
CLONE_DIR = Path("/content/socioswarm_repo")

if GIT_REPO:
    if CLONE_DIR.exists():
        subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=False)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH, GIT_REPO, str(CLONE_DIR)],
            check=True,
        )
    REPO_ROOT = CLONE_DIR
else:
    REPO_ROOT = Path("/content/socioswarm_repo")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("cwd =", Path.cwd())
assert (REPO_ROOT / "scripts" / "run_plantswarm.py").is_file(), "Clone or upload the repo; scripts/run_plantswarm.py not found."


## 4 — Models: IDs, download & load (everything in Colab)

Edit **`PLANTSWARM_BACKBONE`** to match the checkpoint you serve with **vLLM** for `run_plantswarm`. **`DEMO_VLM_ID`** is for the Transformers smoke test only.

This cell **pre-downloads** weights via `snapshot_download` and runs a short **pipeline + generate** on `DEMO_VLM_ID` so models are loaded in-notebook. Trim **`REPLICA_BACKBONES`** in the next cell if downloads are too large.


In [ ]:
# -*- coding: utf-8 -*-
"""HF model IDs, snapshot_download (synced to §1 cache), then Transformers load + smoke inference."""
from __future__ import annotations

import gc
import os

os.environ.setdefault("PYTHONIOENCODING", "utf-8")

if not os.environ.get("HUGGINGFACE_HUB_CACHE"):
    raise RuntimeError(
        "Hugging Face cache not set. Run §1 first so HF_HOME / HUGGINGFACE_HUB_CACHE are configured."
    )

# --- Edit to match vLLM --model and paper experiments --------------------------------
PLANTSWARM_BACKBONE = "Qwen/Qwen3-VL-8B-Instruct"
DEMO_VLM_ID = "Qwen/Qwen3-VL-4B-Thinking"

# Replica list in YAML; set to [] to skip large extra downloads on small GPUs
REPLICA_BACKBONES = [
    "microsoft/Phi-4-multimodal-instruct",
    "Qwen/Qwen2.5-VL-7B-Instruct",
]

# --- Download weights into §1 cache (resumable) --------------------------------------
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

_all = [PLANTSWARM_BACKBONE, DEMO_VLM_ID, *REPLICA_BACKBONES]
seen: set[str] = set()
for repo_id in tqdm(_all, desc="snapshot_download"):
    if repo_id in seen:
        continue
    seen.add(repo_id)
    snapshot_download(repo_id=repo_id, resume_download=True)
    print("snapshot ok:", repo_id)

# --- Load demo VLM in-notebook (validates cache + GPU) --------------------------------
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, pipeline

_messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG",
            },
            {"type": "text", "text": "What animal is on the candy?"},
        ],
    },
]

_bf16 = bool(
    torch.cuda.is_available()
    and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
)
_dtype = torch.bfloat16 if _bf16 else torch.float16

print("\n--- pipeline smoke:", DEMO_VLM_ID, "---")
pipe = pipeline(
    "image-text-to-text",
    model=DEMO_VLM_ID,
    device_map="auto",
    trust_remote_code=True,
    model_kwargs={"torch_dtype": _dtype},
)
print("pipeline:", pipe(text=_messages))

print("\n--- generate smoke:", DEMO_VLM_ID, "---")
processor = AutoProcessor.from_pretrained(DEMO_VLM_ID, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    DEMO_VLM_ID,
    torch_dtype=_dtype,
    device_map="auto",
    trust_remote_code=True,
)
inputs = processor.apply_chat_template(
    _messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=40)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True))

del pipe, model, processor, inputs, outputs
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n[OK] Hub snapshots synced; demo VLM loaded and released. PLANTSWARM_BACKBONE =", PLANTSWARM_BACKBONE)


## 5 — Pipeline YAML (ties config to `PLANTSWARM_BACKBONE`)

Writes `/content/_plantswarm_colab_config.yaml` with `vllm_base_url`, `tfds_data_dir`, and **`model.backbone` = `PLANTSWARM_BACKBONE`**.


In [ ]:
# -*- coding: utf-8 -*-
import os
from pathlib import Path

import yaml

if "PLANTSWARM_BACKBONE" not in globals():
    raise RuntimeError("Run §4 (models) before this cell so PLANTSWARM_BACKBONE / REPLICA_BACKBONES exist.")

CONFIG_PATH = Path("configs/cyag_directory.yaml")
assert CONFIG_PATH.is_file(), CONFIG_PATH

SUBSET = 50
ROUTING_SUBSET = 50
VLLM_BASE_URL = "http://127.0.0.1:8000/v1"

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg.setdefault("model", {})["vllm_base_url"] = VLLM_BASE_URL
cfg["model"]["backbone"] = PLANTSWARM_BACKBONE
cfg["model"]["replica_backbones"] = list(REPLICA_BACKBONES)
if cfg.get("data"):
    cfg["data"]["tfds_data_dir"] = str(Path(os.environ["TFDS_DATA_DIR"]))

CONFIG_RUN = Path("/content/_plantswarm_colab_config.yaml")
CONFIG_RUN.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
print("Wrote", CONFIG_RUN)
print("backbone          =", PLANTSWARM_BACKBONE)
print("replica_backbones =", REPLICA_BACKBONES)
print("vllm_base_url     =", VLLM_BASE_URL)


## 6 — Run all evaluation scripts (one cell)

Needs **vLLM** (or compatible API) reachable at `VLLM_BASE_URL` for steps 01–03. Logs: `RESULTS_DIR/step_logs/`.


In [ ]:
# -*- coding: utf-8 -*-
"""
Run all evaluation scripts sequentially (same order as scripts/run_experiment_bundle.sh).
UTF-8 everywhere; results under RESULTS_DIR.
"""
from __future__ import annotations

import compileall
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("PYTHONIOENCODING", "utf-8")

# Guard: run §1 (paths), §5 (config) before this cell
for _name in ("RESULTS_DIR", "CONFIG_RUN", "SUBSET", "ROUTING_SUBSET"):
    if _name not in globals():
        raise RuntimeError(
            f"Missing required global `{_name}` — run §1–§5 in order before §6."
        )


REPO_ROOT = Path.cwd().resolve()
PYTHON = sys.executable
STEP_LOG = RESULTS_DIR / "step_logs"
STEP_LOG.mkdir(parents=True, exist_ok=True)


def run_step(name: str, argv: list[str]) -> None:
    log_path = STEP_LOG / f"{name}.log"
    print("\n", "=" * 72, f"\n>>> [{name}]\n", "=" * 72, sep="")
    print("CMD:", " ".join(argv))
    with open(log_path, "w", encoding="utf-8") as log:
        p = subprocess.run(
            argv,
            cwd=str(REPO_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        log.write(p.stdout)
        print(p.stdout[-8000:] if len(p.stdout) > 8000 else p.stdout)
        if p.returncode != 0:
            raise RuntimeError(f"Step {name} failed with exit code {p.returncode}; see {log_path}")


def opt_subset() -> list[str]:
    if SUBSET:
        return ["--subset", str(SUBSET)]
    return []


# 0) Byte-compile packages (matches scripts/smoke_test.sh compileall scope)
pkgs = ["agents", "plantswarm", "baselines", "ablations", "calibration", "bias", "utils", "scripts", "data"]
for pkg in pkgs:
    p = REPO_ROOT / pkg
    if p.is_dir():
        compileall.compile_dir(str(p), quiet=1)
print("[compileall] OK for:", ", ".join(pkgs))

cfg_arg = ["--config", str(CONFIG_RUN)]
out_arg = ["--output_dir", str(RESULTS_DIR)]

# 01 PlantSwarm main
run_step(
    "01_plantswarm_main",
    [PYTHON, "scripts/run_plantswarm.py", *cfg_arg, *opt_subset(), "--orchestrator", "autogen_swarm", *out_arg],
)

# 02 Baselines
run_step(
    "02_baselines",
    [PYTHON, "scripts/run_baselines.py", *cfg_arg, *opt_subset(), *out_arg],
)

# 03 Ablations
run_step(
    "03_ablations",
    [PYTHON, "scripts/run_ablations.py", *cfg_arg, *opt_subset(), *out_arg],
)

# 04 Calibration
run_step(
    "04_calibration",
    [
        PYTHON,
        "scripts/run_calibration.py",
        *cfg_arg,
        "--predictions",
        str(RESULTS_DIR / "plantswarm_predictions.jsonl"),
        *out_arg,
    ],
)

# 05 Routing analysis
run_step(
    "05_routing_analysis",
    [
        PYTHON,
        "scripts/run_routing_analysis.py",
        *cfg_arg,
        "--subset",
        str(ROUTING_SUBSET),
        *out_arg,
    ],
)

# 06 Bias analysis
run_step(
    "06_bias_analysis",
    [
        PYTHON,
        "scripts/run_bias_analysis.py",
        *cfg_arg,
        "--traces",
        str(RESULTS_DIR / "traces" / "plantswarm_traces.jsonl"),
        "--predictions",
        str(RESULTS_DIR / "plantswarm_predictions.jsonl"),
        *out_arg,
    ],
)

# 07 Sync LaTeX tables from metrics JSON
subset_hint = str(SUBSET) if SUBSET else "full"
run_step(
    "07_sync_latex",
    [
        PYTHON,
        "scripts/sync_latex_metrics.py",
        "--results-dir",
        str(RESULTS_DIR),
        "--latex-dir",
        str(REPO_ROOT / "plantswarm" / "latex"),
        "--subset-hint",
        subset_hint,
    ],
)

# 08 PDF (optional: needs latexmk + TeX; often missing on Colab)
build_pdf = shutil.which("latexmk") is not None
if build_pdf:
    run_step(
        "08_build_pdf",
        [
            "bash",
            str(REPO_ROOT / "scripts" / "build_latex_pdf.sh"),
            "--latex-dir",
            str(REPO_ROOT / "plantswarm" / "latex"),
            "--main-tex",
            "acl_latex.tex",
            "--results-dir",
            str(RESULTS_DIR),
        ],
    )
else:
    print("[SKIP] latexmk not found — install TeX or build PDF locally from synced auto_*.tex")

print("\nDONE. All outputs under:", RESULTS_DIR)
print("Per-step logs:", STEP_LOG)



## 7 — Optional: TFDS / PlantDoc verify

Uncomment the next cell to run verification scripts.


In [ ]:
# -*- coding: utf-8 -*-
# import subprocess, sys
# subprocess.check_call([sys.executable, "scripts/verify_tfds_plant_village.py", "--max-examples", "16"])
# subprocess.check_call([sys.executable, "scripts/verify_plantdoc_repo.py", "/content/drive/MyDrive/PlantDoc-Dataset", "--split", "train"])

